In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import pyopenms as oms
from scipy.signal import find_peaks, peak_widths
import scipy.signal as signal
import scipy.optimize as opt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
%matplotlib inline

'''
This section loads the data file and plots the TIC 
'''
exp = oms.MSExperiment()
oms.MzMLFile().load('enter path for data mzML here', exp)

# Initialize an empty TIC chromatogram
tic = oms.MSChromatogram()

# Iterate through chromatograms to find the TIC
for chromatogram in exp.getChromatograms():
    if "TIC" in chromatogram.getNativeID():
        tic = chromatogram
        break

# Use the get_peaks method to retrieve retention time and intensity values
time_array, intensity_array = tic.get_peaks()

# Convert retention time from seconds to minutes
time_array_minutes = time_array / 60

# Plot the TIC with retention time in minutes
plt.figure(figsize=(10, 6))
plt.plot(time_array_minutes, intensity_array, label="TIC")

# Add more ticks to the x-axis
plt.xticks(np.arange(0, np.max(time_array_minutes), step=5))  # Adjust the step size to add more ticks

# Show gridlines (optional)
plt.grid(True)

# Add labels and title
plt.xlabel("Retention Time (minutes)")
plt.ylabel("Intensity")
plt.title("Total Ion Chromatogram (TIC)")
plt.legend()

# Display the plot
plt.show()


In [ ]:
'''
This section takes the TIC, smoothes the baseline and identifies the TIC peaks. 
The peaks are saved in an excel spreadsheet with retention times, peak start and end time, peak width, peak area, max intensity and peak area percentage.
The peak detection is presented in a png file, showing the baseline smoothing, thresholds and selected peak boundaries.
'''
# ---------------------------- Initialization Section ----------------------------
# Adjustable Parameters for baseline smoothing, thresholding and peak picking
# Values were adjusted for known samples, for ideal conditions
min_rt_cutoff_minutes = 8 # Minimum retention time cutoff in minutes, Peaks before this time are ignored
max_rt_cutoff_minutes = 35  # Maximum retention time cutoff in minutes, Peaks after this time are ignored
window_size = 5  # Moving average filter size for smoothing the chromatogram
signal_to_noise_ratio = 0.5 # Signal-to-noise ratio for pyOpenMS peak picker
sgolay_frame_length = 15  # Frame length for the Savitzky-Golay filter
sgolay_polynomial_order = 5  # Polynomial order for Savitzky-Golay filter
spacing_difference_gap = 1.5  # Maximum allowed difference between peak spacing in pyOpenMS
prominence_threshold = 10000  # Minimum prominence for SciPy peak detection
min_distance_between_peaks_minutes = 0.3 # Minimum distance between peaks in minutes
initial_refinement_window_seconds = 10  # Initial window size in seconds for apex refinement
final_refinement_window_seconds = 15 # Window size in seconds for final peak boundary refinement
area_percentage_threshold = 1 # Minimum area percentage threshold for filtering insignificant peaks
large_peak_intensity_threshold = 4000000000  # Threshold to classify large peaks, adjust based on what instrument used for analysis
intensity_threshold_fraction = 0.01 # Fraction of peak's maximum intensity for determining the start boundary of large peaks
scipy_peak_count = 30  # Number of unique SciPy peaks to add to pyOpenMS peaks

# Baseline adjustment parameters
baseline_smoothing_window = 100 # Larger window leads to smoother baseline
baseline_polynomial_order = 1 # Lower polynomial order leads to simpler baseline
peak_mask_threshold = 0.002 # Higher threshold for peak exclusion
num_iterations = 2  # Fewer iterations for lighter correction

output_filtered_peaks_csv = "filtered_peaks_combined.csv"  # File to save filtered peaks
output_all_peaks_csv = "all_peaks_combined.csv"  # File to save all detected peaks
output_plot_filename = "chromatogram_detected_peaks_combined.png"  # File to save the chromatogram plot

# -------------------------------------------------------------------------------
# Load and Filter Chromatogram Data
min_rt_cutoff = min_rt_cutoff_minutes * 60  # Convert minutes to seconds
max_rt_cutoff = max_rt_cutoff_minutes * 60  # Convert minutes to seconds
time_array, intensity_array = tic.get_peaks()  # Retrieve time and intensity arrays

filtered_time_array = time_array[(time_array >= min_rt_cutoff) & (time_array <= max_rt_cutoff)]
filtered_intensity_array = intensity_array[(time_array >= min_rt_cutoff) & (time_array <= max_rt_cutoff)]

if baseline_smoothing_window % 2 == 0:
    baseline_smoothing_window += 1

# Baseline Correction
baseline = savgol_filter(filtered_intensity_array, window_length=baseline_smoothing_window, polyorder=baseline_polynomial_order)

# Refine the baseline iteratively
for _ in range(num_iterations):
    residual = filtered_intensity_array - baseline
    peak_mask = residual > (peak_mask_threshold * residual.max())
    non_peak_time = filtered_time_array[~peak_mask]
    non_peak_intensity = filtered_intensity_array[~peak_mask]

    if len(non_peak_time) > baseline_smoothing_window:
        baseline = savgol_filter(non_peak_intensity, window_length=baseline_smoothing_window, polyorder=baseline_polynomial_order)
        baseline = np.interp(filtered_time_array, non_peak_time, baseline)  # Interpolate back to full array length

# Correct the intensity array
corrected_intensity_array = filtered_intensity_array - baseline
corrected_intensity_array[corrected_intensity_array < 0] = 0  
filtered_intensity_array = corrected_intensity_array

# Smooth Data
smoothed_intensity_array = np.convolve(filtered_intensity_array, np.ones(window_size) / window_size, mode='same')

# Detect Peaks with PyOpenMS
smoothed_tic = oms.MSChromatogram()
smoothed_tic.set_peaks((filtered_time_array, smoothed_intensity_array))

peak_picker = oms.PeakPickerChromatogram()
params = oms.Param()
params.setValue("signal_to_noise", signal_to_noise_ratio)
params.setValue("sgolay_frame_length", sgolay_frame_length)
params.setValue("sgolay_polynomial_order", sgolay_polynomial_order)
params.setValue("spacing_difference_gap", spacing_difference_gap)
peak_picker.setParameters(params)

picked_chrom = oms.MSChromatogram()
peak_picker.pickChromatogram(smoothed_tic, picked_chrom)
picked_time_array, picked_intensity_array = picked_chrom.get_peaks()

refined_pyopenms_peaks = []
for i in range(len(picked_time_array)):
    peak_rt = picked_time_array[i]
    fit_window_indices = (filtered_time_array > peak_rt - initial_refinement_window_seconds) & (
        filtered_time_array < peak_rt + initial_refinement_window_seconds)
    fit_time = filtered_time_array[fit_window_indices]
    fit_intensity = filtered_intensity_array[fit_window_indices]
    if len(fit_time) > 0:
        max_idx = np.argmax(fit_intensity)
        refined_peak_rt = fit_time[max_idx]
        refined_peak_intensity = fit_intensity[max_idx]
        refined_pyopenms_peaks.append((refined_peak_rt, refined_peak_intensity))

refined_pyopenms_peaks_df = pd.DataFrame(refined_pyopenms_peaks, columns=['Retention Time', 'Intensity'])

# Detect Additional Peaks with SciPy (PyOpenMS does not always detect small peaks so this additional method was added)
# Detect peaks with SciPy using the prominence threshold
peaks_scipy, _ = signal.find_peaks(smoothed_intensity_array, prominence=prominence_threshold)
time_array_picked_scipy = filtered_time_array[peaks_scipy]
intensity_array_picked_scipy = smoothed_intensity_array[peaks_scipy]

# Filter SciPy peaks to exclude those already detected by pyOpenMS
filtered_scipy_peaks = []
min_distance = min_distance_between_peaks_minutes * 60  # Convert minutes to seconds
for i in range(len(time_array_picked_scipy)):
    scipy_rt = time_array_picked_scipy[i]
    scipy_intensity = intensity_array_picked_scipy[i]
    overlap = False

    # Check for overlap with pyOpenMS peaks
    for py_rt in refined_pyopenms_peaks_df['Retention Time']:
        if abs(py_rt - scipy_rt) < min_distance:
            overlap = True
            break

    # Check for overlap with already selected SciPy peaks
    for final_rt, _ in filtered_scipy_peaks:
        if abs(final_rt - scipy_rt) < min_distance:
            overlap = True
            break

    if not overlap:
        filtered_scipy_peaks.append((scipy_rt, scipy_intensity))

# Convert filtered SciPy peaks to a DataFrame
filtered_scipy_peaks_df = pd.DataFrame(
    filtered_scipy_peaks, columns=['Retention Time', 'Intensity']
).sort_values(by='Retention Time')  # Keep SciPy peaks sorted by retention time

# Select the top N additional peaks (defined by scipy_peak_count)
selected_scipy_peaks_df = filtered_scipy_peaks_df.sort_values(by='Intensity', ascending=False).head(scipy_peak_count)

# Combine PyOpenMS and selected SciPy peaks
final_peaks_df = pd.concat([refined_pyopenms_peaks_df, selected_scipy_peaks_df], ignore_index=True).sort_values(
    by='Retention Time'
)

# Output the total number of peaks selected
print(f"Current number of peaks selected: {len(final_peaks_df)}")

# Apex Refinement (True Maxima), make sure the peak is actually identified as peak at its peak
final_corrected_peaks = []
for _, row in final_peaks_df.iterrows():
    peak_rt = row['Retention Time']
    start_idx = max(0, np.searchsorted(filtered_time_array, peak_rt - final_refinement_window_seconds))
    end_idx = min(len(filtered_time_array) - 1, np.searchsorted(filtered_time_array, peak_rt + final_refinement_window_seconds))
    corrected_max_idx = start_idx + np.argmax(filtered_intensity_array[start_idx:end_idx + 1])
    corrected_peak_rt = filtered_time_array[corrected_max_idx]
    corrected_peak_intensity = filtered_intensity_array[corrected_max_idx]
    final_corrected_peaks.append((corrected_peak_rt, corrected_peak_intensity))

final_corrected_peaks_df = pd.DataFrame(final_corrected_peaks, columns=['Retention Time', 'Intensity'])

# Split Peaks into Large and Small
# Different boundary detection is needed for different peak sizes to ensure optimal peak picking
large_peaks = final_corrected_peaks_df[final_corrected_peaks_df['Intensity'] >= large_peak_intensity_threshold]
small_peaks = final_corrected_peaks_df[final_corrected_peaks_df['Intensity'] < large_peak_intensity_threshold]

# Refined boundaries for large peaks
large_peaks_with_refined_boundaries = []
for _, row in large_peaks.iterrows():
    peak_rt = row['Retention Time']
    intensity = row['Intensity']

    # Extend boundaries more generously to include the entire large peak
    start_idx = max(0, np.searchsorted(filtered_time_array, peak_rt - final_refinement_window_seconds * 2))
    end_idx = min(len(filtered_time_array) - 1, np.searchsorted(filtered_time_array, peak_rt + final_refinement_window_seconds * 2))
    local_time = filtered_time_array[start_idx:end_idx + 1]
    local_intensity = filtered_intensity_array[start_idx:end_idx + 1]

    # Calculate the slope
    slope = np.gradient(local_intensity, local_time)

    # Define intensity threshold to refine the start boundary
    intensity_threshold_fraction = 0.01  
    intensity_threshold = intensity * intensity_threshold_fraction

    # Find the first point where intensity exceeds the threshold
    valid_intensity_indices = np.where(local_intensity >= intensity_threshold)[0]

    # Combine slope and intensity to define start and end boundaries
    if len(valid_intensity_indices) > 0:
        refined_start_idx = valid_intensity_indices[0]
        start_rt = local_time[refined_start_idx]

        # End boundary based on slope analysis 
        zero_slope_indices = np.where(np.isclose(slope, 0, atol=0.001))[0]
        if len(zero_slope_indices) > 0:
            end_rt = local_time[zero_slope_indices[-1]]
        else:
            end_rt = peak_rt + final_refinement_window_seconds * 2
    else:
        # Fallback to wider boundaries if no valid region is found
        start_rt = peak_rt - final_refinement_window_seconds * 2
        end_rt = peak_rt + final_refinement_window_seconds * 2

    large_peaks_with_refined_boundaries.append((peak_rt, intensity, start_rt, end_rt))

# Exclude small peaks within large peak boundaries
refined_small_peaks = []
for _, row in small_peaks.iterrows():
    peak_rt = row['Retention Time']
    intensity = row['Intensity']
    within_large_peak = False
    for large_peak in large_peaks_with_refined_boundaries:
        large_start_rt, large_end_rt = large_peak[2], large_peak[3]
        if large_start_rt <= peak_rt <= large_end_rt:
            within_large_peak = True
            break
    if not within_large_peak:
        refined_small_peaks.append((peak_rt, intensity, peak_rt - initial_refinement_window_seconds, peak_rt + initial_refinement_window_seconds))

# Combine all peaks with refined boundaries
all_peaks_with_boundaries = large_peaks_with_refined_boundaries + refined_small_peaks

# Deduplicate peaks to ensure each retention time is counted only once (with a tolerance for proximity)
deduplicated_peaks = {}

# Define the proximity threshold 
retention_time_tolerance = 0.01 * 60  # Convert minutes to seconds

for peak in all_peaks_with_boundaries:
    retention_time = peak[0]  # Retention Time
    found_match = False

    for existing_rt in list(deduplicated_peaks.keys()):
        if abs(existing_rt - retention_time) <= retention_time_tolerance:
            # If a match is found, compare intensities and keep the higher intensity peak
            if peak[1] > deduplicated_peaks[existing_rt][1]:  # Compare Intensity
                deduplicated_peaks[existing_rt] = peak
            found_match = True
            break

    if not found_match:
        deduplicated_peaks[retention_time] = peak

# Convert deduplicated peaks back to a list
all_peaks_with_boundaries = list(deduplicated_peaks.values())

# Create a DataFrame from the deduplicated peaks
filtered_df = pd.DataFrame(all_peaks_with_boundaries, columns=['Retention Time', 'Intensity', 'Peak Start', 'Peak End'])

# Convert boundaries to minutes for plotting
filtered_df['Peak Start (minutes)'] = filtered_df['Peak Start'] / 60
filtered_df['Peak End (minutes)'] = filtered_df['Peak End'] / 60

# Calculate peak width, area, and area percentage
peak_data = []
for _, row in filtered_df.iterrows():
    start_idx = np.searchsorted(filtered_time_array, row['Peak Start'])
    end_idx = np.searchsorted(filtered_time_array, row['Peak End'])
    
    # Calculate peak area using the trapezoidal rule
    peak_area = np.trapz(filtered_intensity_array[start_idx:end_idx], filtered_time_array[start_idx:end_idx]) / 60 
    peak_width = row['Peak End (minutes)'] - row['Peak Start (minutes)']  # Width in minutes
    
    # Append detailed peak data
    peak_data.append((
        row['Retention Time']/60, 
        row['Peak Start (minutes)'], 
        row['Peak End (minutes)'], 
        peak_width, 
        peak_area, 
        row['Intensity']
    ))

# Create a DataFrame from the peak data
peaks_df = pd.DataFrame(peak_data, columns=[
    'Retention Time', 'Peak Start (minutes)', 'Peak End (minutes)', 'Peak Width (minutes)', 'Peak Area', 'Max Intensity'
])

# Calculate the area percentage of each peak relative to the largest peak area
max_peak_area = peaks_df['Peak Area'].max()
peaks_df['Area Percentage (%)'] = (peaks_df['Peak Area'] / max_peak_area) * 100

# Filter peaks based on area percentage threshold to only keep significant peaks
filtered_df = peaks_df[peaks_df['Area Percentage (%)'] >= area_percentage_threshold]

# Output the final number of peaks after all refinements
print(f"Total number of peaks selected after refinement: {len(filtered_df)}")

# Visualization with Baseline Correction and Final Peaks Highlighted
plt.figure(figsize=(12, 6))

# Plot raw data with baseline
plt.plot(filtered_time_array / 60, intensity_array[(time_array >= min_rt_cutoff) & (time_array <= max_rt_cutoff)], label='Raw Data', color='blue')
plt.plot(filtered_time_array / 60, baseline, label='Estimated Baseline', color='orange', linestyle='--')
plt.plot(filtered_time_array / 60, corrected_intensity_array, label='Baseline-Corrected Data', color='black')

# Add intensity threshold line
plt.axhline(y=large_peak_intensity_threshold, color='red', linestyle='--', label='Intensity Threshold')

# Calculate and add area percentage threshold line
area_threshold = area_percentage_threshold / 100 * max_peak_area  # Threshold area value
plt.axhline(y=area_threshold, color='teal', linestyle='-.', label='Area Percentage Threshold')

# Add corrected detected peaks (blue crosses)
plt.plot(refined_pyopenms_peaks_df['Retention Time'] / 60, refined_pyopenms_peaks_df['Intensity'], 'bx', label='Corrected Detected Peaks')

# Add final selected peaks (red dots)
plt.plot(filtered_df['Retention Time'], filtered_df['Max Intensity'], 'ro', label='Final Selected Peaks')

# Add peak boundaries (purple and green dashed lines)
for _, row in filtered_df.iterrows():
    plt.vlines(row['Peak Start (minutes)'], ymin=0, ymax=row['Max Intensity'], color='purple', linestyles='dashed', label='Peak Start Boundary' if _ == 0 else "")
    plt.vlines(row['Peak End (minutes)'], ymin=0, ymax=row['Max Intensity'], color='green', linestyles='dashed', label='Peak End Boundary' if _ == 0 else "")

# Label the plot
plt.xlabel('Retention Time (minutes)')
plt.ylabel('Intensity / Area')
plt.title('TIC with Enhanced Baseline Correction and Final Peaks')
plt.legend()
plt.grid(True)
plt.savefig(output_plot_filename)
plt.show()

# Save the filtered peaks DataFrame to CSV
filtered_df.to_csv(output_filtered_peaks_csv, index=False)


In [ ]:
'''
This section takes the peviously identified TIC peaks and plots the averaged MS spectrum for each peak retention time window seperately
'''
# Initialization
# This part uses the 'filtered_df' from above
mz_precision = 0.001  # Precision to round m/z values to align spectra
intensity_threshold = 0  # Minimum intensity to filter spectra, this is an option for thresholding already at this data processing step if required

# Sort the filtered_df by 'Retention Time (minutes)' from lowest to highest
filtered_df = filtered_df.sort_values(by='Retention Time').reset_index(drop=True)

# Create a SpectraMerger object
merger = oms.SpectraMerger()

for idx, row in filtered_df.iterrows():
    peak_start_sec = row['Peak Start (minutes)'] * 60  
    peak_end_sec = row['Peak End (minutes)'] * 60      
    retention_time_min = row['Retention Time']

    # Collect spectra within the peak time window
    spectra_in_window = [spectrum for spectrum in exp if peak_start_sec <= spectrum.getRT() <= peak_end_sec]

    # Check if there are spectra in the window
    if not spectra_in_window:
        print(f"No spectra found for peak {idx+1}.")
        continue

    # Filter spectra by intensity threshold
    filtered_spectra = []
    for spectrum in spectra_in_window:
        mz_array, intensity_array = spectrum.get_peaks()
        if max(intensity_array) >= intensity_threshold:
            filtered_spectra.append(spectrum)

    # Check if there are enough spectra to proceed
    if not filtered_spectra:
        print(f"No spectra passed the intensity threshold for peak {idx+1}.")
        continue

    # Align the filtered spectra by rounding m/z values to defined precision
    aligned_intensity = {}
    for spectrum in filtered_spectra:
        mz_array, intensity_array = spectrum.get_peaks()
        
        # Round the m/z values to the specified precision
        rounded_mz_array = np.round(mz_array / mz_precision) * mz_precision
        
        # Aggregate intensities at the rounded m/z values
        for mz, intensity in zip(rounded_mz_array, intensity_array):
            if mz not in aligned_intensity:
                aligned_intensity[mz] = intensity
            else:
                aligned_intensity[mz] += intensity

    # Create arrays from the aligned data
    aligned_mz_array = np.array(sorted(aligned_intensity.keys()))
    aligned_intensity_array = np.array([aligned_intensity[mz] for mz in aligned_mz_array])

    # Create a DataFrame for the aligned summed spectrum
    peak_spectrum_df = pd.DataFrame({
        'm/z': aligned_mz_array,
        'Intensity': aligned_intensity_array
    }).sort_values(by='m/z').reset_index(drop=True)

    # Store the DataFrame in a variable named after the peak retention time
    var_name = f"peak_{idx+1}_summed_df"
    globals()[var_name] = peak_spectrum_df  # Create a variable with the name 'peak_#_summed_df'

    # Save the DataFrame to a CSV file
    peak_spectrum_df.to_csv(f"peak_spectrum_summed_{idx+1}.csv", index=False)

    # Plot the aligned summed spectrum
    plt.figure(figsize=(10, 6))
    plt.plot(peak_spectrum_df['m/z'], peak_spectrum_df['Intensity'], label=f"Peak {idx+1} at RT {retention_time_min:.2f} min")
    plt.title(f"Aligned Summed Spectrum for Peak {idx+1} at RT {retention_time_min:.2f} min")
    plt.xlabel("m/z")
    plt.ylabel("Intensity")
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Aligned summed spectrum for peak {idx+1} stored in {var_name}.")

In [ ]:
'''
This section takes the peviously averaged MS spectra and applies the PyOpenMS peak_picker function to identify all peaks in the spectrum.
Then the peaks are being counted under consideration of different thresholds and saved to a final_overview excel spreadsheet. 
The final peak count is put in relation with the retention time peak in the chromatogram, the most intense peak observed and the heaviest peak observed in that thresholding window. 
While the final_overview spreadsheet has all the data required for further data analysis, a final_overview_extended spreadsheet is also exported, with all in the process calculated values. 
'''

# Initialize the peak picker for high-resolution spectra
peak_picker = oms.PeakPickerHiRes()

# List to hold final peak overview data 
overview_data_extended = []

# Define intensity thresholds 
intensity_thresholds = [0, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]
threshold_keys = [
    "No Threshold", "Threshold 1e4", "Threshold 1e5",
    "Threshold 1e6", "Threshold 1e7", "Threshold 1e8", "Threshold 1e9", "Threshold 1e10"
]

# Loop through each averaged spectrum dataframe created before
for idx in range(1, len(filtered_df) + 1):  # assumes 'peak_#_summed_df' exist
    # Retrieve the spectrum DataFrame
    spectrum_df = globals()[f'peak_{idx}_summed_df']

    # Build and peak-pick spectrum
    exp = oms.MSExperiment()
    spectrum = oms.MSSpectrum()
    spectrum.set_peaks((spectrum_df['m/z'].values, spectrum_df['Intensity'].values))
    exp.addSpectrum(spectrum)

    picked_spectrum = oms.MSSpectrum()
    peak_picker.pick(spectrum, picked_spectrum)

    # Peak-picked dataframe
    mz_array, intensity_array = picked_spectrum.get_peaks()
    picked_peaks_df = pd.DataFrame({'m/z': mz_array, 'Intensity': intensity_array})

    # Containers for per-threshold outputs
    peak_counts = {key: 0 for key in threshold_keys}
    heaviest_mz = {key: float('nan') for key in threshold_keys}
    most_intense_mz = {key: float('nan') for key in threshold_keys}

    # Sort once (helps deterministic idxmax ties; optional)
    if not picked_peaks_df.empty:
        picked_peaks_df = picked_peaks_df.sort_values(by='Intensity', ascending=False).reset_index(drop=True)

    # "No Threshold" (all picked peaks)
    peak_counts["No Threshold"] = len(picked_peaks_df)
    if not picked_peaks_df.empty:
        heaviest_mz["No Threshold"] = float(picked_peaks_df['m/z'].max())
        most_intense_mz["No Threshold"] = float(picked_peaks_df.loc[picked_peaks_df['Intensity'].idxmax(), 'm/z'])

    # Thresholded stats
    for threshold, key in zip(intensity_thresholds[1:], threshold_keys[1:]):  # skip 0; handled above
        if picked_peaks_df.empty:
            peak_counts[key] = 0
            heaviest_mz[key] = float('nan')
            most_intense_mz[key] = float('nan')
        else:
            filtered_peaks_df = picked_peaks_df[picked_peaks_df['Intensity'] >= threshold]
            peak_counts[key] = int(len(filtered_peaks_df))
            heaviest_mz[key] = float(filtered_peaks_df['m/z'].max()) if not filtered_peaks_df.empty else float('nan')
            most_intense_mz[key] = (
                float(filtered_peaks_df.loc[filtered_peaks_df['Intensity'].idxmax(), 'm/z'])
                if not filtered_peaks_df.empty else float('nan')
            )

    # Pull RT window metadata from your chromatogram table
    row = filtered_df.iloc[idx - 1]
    retention_time = row['Retention Time']
    peak_start = row['Peak Start (minutes)']
    peak_end = row['Peak End (minutes)']
    peak_area_chrom = row['Peak Area']  # chromatogram/time window area

    if not spectrum_df.empty:
        raw_max_idx = spectrum_df['Intensity'].idxmax()
        raw_max_mz  = float(spectrum_df.loc[raw_max_idx, 'm/z'])
        raw_max_int = float(spectrum_df.loc[raw_max_idx, 'Intensity'])
    else:
        raw_max_mz = float('nan')
        raw_max_int = float('nan')

    # Most intense picked peak (no threshold)
    if not picked_peaks_df.empty:
        top_idx = picked_peaks_df['Intensity'].idxmax()
        most_intense_peak_mz  = float(picked_peaks_df.loc[top_idx, 'm/z'])
        most_intense_peak_int = float(picked_peaks_df.loc[top_idx, 'Intensity'])
    else:
        most_intense_peak_mz = float('nan')
        most_intense_peak_int = float('nan')

    # Dynamic columns for extended version
    count_cols = {f'Number of Picked Peaks ({key})': peak_counts[key] for key in threshold_keys}
    heavy_cols = {f'Heaviest m/z ({key})': heaviest_mz[key] for key in threshold_keys}
    intense_cols = {f'Most Intense m/z ({key})': most_intense_mz[key] for key in threshold_keys}

    # Assemble extended row
    overview_data_extended.append({
        'Retention Time (minutes)': retention_time,
        'Peak Start (minutes)': peak_start,
        'Peak End (minutes)': peak_end,
        'Peak Area (Chromatogram)': peak_area_chrom,

        'Max Intensity m/z (raw spectrum)': raw_max_mz,
        'Max Intensity (raw spectrum)': raw_max_int,

        'Most Intense Peak m/z (picked, no threshold)': most_intense_peak_mz,
        'Most Intense Peak Intensity (picked, no threshold)': most_intense_peak_int,

        **heavy_cols,
        **count_cols,
        **intense_cols,  # present only in the extended file
    })

# Create EXTENDED dataframe and save
final_overview_extended = pd.DataFrame(overview_data_extended)
print("final_overview_extended:")
print(final_overview_extended)
final_overview_extended.to_csv("final_overview_extended.csv", index=False)

# Column order helper for streamlined final_overview file
base_cols = [
    'Retention Time (minutes)',
    'Peak Start (minutes)',
    'Peak End (minutes)',
    'Peak Area (Chromatogram)',
    'Most Intense Peak m/z (picked, no threshold)',
    'Most Intense Peak Intensity (picked, no threshold)',
]

no_th_cols = [
    'Heaviest m/z (No Threshold)',
    'Number of Picked Peaks (No Threshold)',
]

# Then for each threshold > No Threshold: heaviest m/z then number of peaks
ordered_threshold_cols = []
for key in threshold_keys[1:]:
    ordered_threshold_cols.append(f'Heaviest m/z ({key})')
    ordered_threshold_cols.append(f'Number of Picked Peaks ({key})')

# Build the final ordered column list
final_cols = base_cols + no_th_cols + ordered_threshold_cols

# Subset + reorder; missing columns (if any) will be created as NaN to avoid KeyErrors
for col in final_cols:
    if col not in final_overview_extended.columns:
        final_overview_extended[col] = float('nan')

final_overview = final_overview_extended[final_cols].copy()
print("final_overview:")
print(final_overview)
final_overview.to_csv("final_overview.csv", index=False)
